# 财经流媒体心理暗示与情绪操控 24/7 监控系统 (Google Colab 版)

本 Notebook 将监控系统部署在 Google Colab，并通过 **Google Drive 持久化 SQLite 数据库**，确保 7 天滚动基线（UV Z-Score）与 6 小时告警冷却期在会话重连后不丢失。

> **Colab 运行提示**：
> 1. 免费版 Colab 闲置约 90 分钟或连续运行 12 小时后会自动断开；建议在浏览器保持标签页开启，或使用 Colab Pro。
> 2. 若 Colab 海外 IP 抓取国内社区接口被限流，可先运行第 4 步的模拟演练确认推送链路，再根据需要配置代理。

In [ ]:
# 1. 挂载 Google Drive 并定位项目目录
from google.colab import drive
import os, sys

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/stock-sentiment-monitor'
os.makedirs(PROJECT_DIR, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('当前工作目录:', PROJECT_DIR)
print('请确保已将本项目的 src/ 文件夹上传至:', PROJECT_DIR)

In [ ]:
# 2. 安装轻量级依赖 (约 20 秒)
!pip install -q akshare jieba requests

In [ ]:
# 3. 配置监控参数与推送通道
import logging
from src.config import MonitorConfig
from src.monitor import SentimentMonitor

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

config = MonitorConfig(
    db_path=f'{PROJECT_DIR}/monitor_data.db',  # 持久化在 Google Drive
    risk_score_threshold=75.0,
    alert_cooldown_hours=6,
    # 推送配置示例 (二选一，留空则仅在 Colab 控制台打印)
    webhook_type='feishu',       # 可选: feishu | dingtalk | wecom | nonebot
    webhook_url='',              # 填入飞书/钉钉/企业微信 Webhook
    # 若使用 NoneBot2 (OneBot v11 HTTP API):
    nonebot_api_url='',          # 例如: http://你的公网IP:5700/send_group_msg
    nonebot_group_id=0,
    nonebot_token='',
    # 监控标的列表 (留空则自动扫描东财人气热榜前 10)
    watchlist=[]
)

monitor = SentimentMonitor(config)
print('Monitor 初始化完成!')

In [ ]:
# 4. 离线模拟验证 (验证 4 类典型场景评分、公告白名单与推送格式)
results = monitor.run_once(use_mock=True)
for r in results:
    print(f'{r.name}({r.symbol}) | Risk={r.risk_score:.1f} | {r.regime} | UV={r.uv_count} | 同质率={r.homogeneity_ratio:.0%}')

In [ ]:
# 5. 启动全天自适应监控循环 (盘前/盘后 5 分钟一次，盘中 10 分钟，夜间 30 分钟)
monitor.run_forever(use_mock=False)